1. Prepare Dataset

######## Important Note ########
# Change the vocab size and dataset according to your need
# Make sure the vocab size in both tokenizer training and model config are the same
################################

In [1]:
# Import dataset before loading
from datasets import concatenate_datasets, load_dataset

# Custon save location
# save_location = r"D:\LM\BERT_pretrain_practice\load_dataset"

# Load bookcorpus dataset
bookcorpus = load_dataset("bookcorpus", split="train")
# Save to disk at custom location
# bookcorpus.save_to_disk(f"{save_location}/bookcorpus")

d:\Anaconda\envs\bert_pretrain\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
Using the latest cached version of the dataset since bookcorpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\q\.cache\huggingface\datasets\bookcorpus\plain_text\1.0.0\eddee3cae1cc263a431aa98207d4d27fd8a73b0a9742f692af0e6c65afa4d75f (last modified on Mon Nov  3 17:17:07 2025).


In [2]:
# Load wikipedia dataset
# If encountering issue 'Dataset scripts are no longer supported, but found'
#   There might be an issue with the newest huggingface_hub or datasets release
#   Try to downgrade to previous version: pip install datasets==2.16.0
wiki = load_dataset('wikipedia', '20220301.en', split='train')


# Clean wiki dataset to keep only text col
wiki = wiki.remove_columns([col for col in wiki.column_names if col != 'text'])

Using the latest cached version of the dataset since wikipedia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '20220301.en' at C:\Users\q\.cache\huggingface\datasets\wikipedia\20220301.en\2.0.0\d41137e149b2ea90eead07e7e3f805119a8c22dd1d5b61651af8e3e3ee736001 (last modified on Wed Oct 15 13:13:10 2025).


In [3]:
# Compare structure of both datasets, halt if not the same
assert bookcorpus.features.type == wiki.features.type, "Dataset structures are not the same"

In [4]:
# Concatonate both datasets
contatonated_dataset = concatenate_datasets([bookcorpus, wiki])

# Reduced size dataset
small_test_dataset = contatonated_dataset.shuffle(seed=114514).select(range(50000))

2. Train Tokenizer

In [6]:
# Next, we train a tokenizer on the dateset
from tqdm import tqdm
from transformers import BertTokenizerFast

# Tokenizer id here
#   It uses bert-base-uncased and Habana Gaudi platform optimizations
tokenizer_id = "bert-base-uncased"

In [7]:
# Create a generator to dynamically load text data in chunks
def batch_iterator(dataset,batch_size=10000):
    for i in tqdm(range(0, len(dataset), batch_size)):
        yield dataset[i: i + batch_size]["text"]

# Create a tokenizer from the pretrained tokenizer to re-use exisiting special tokens
pretrained_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
# Specify vocab size, reduce for lighter training
#   Original BERT uses 30,522

####### Change vocab size here #######
volcab_size = 6000

# Start the training of tokenizer 
#   Inherit from the pretrained tokenizer to re-use special tokens
#   The training method provided by the transformers library, using WordPiece algorithm
my_bert_tokenizer = pretrained_tokenizer.train_new_from_iterator(
    text_iterator=batch_iterator(small_test_dataset),
    vocab_size=volcab_size
)

# Save the tokenizer to local disk
my_bert_tokenizer.save_pretrained(f"./{tokenizer_id}" + "_volcab_size_" + str(volcab_size))

100%|██████████| 5/5 [00:12<00:00,  2.53s/it]


('./bert-base-uncased_volcab_size_6000\\tokenizer_config.json',
 './bert-base-uncased_volcab_size_6000\\special_tokens_map.json',
 './bert-base-uncased_volcab_size_6000\\vocab.txt',
 './bert-base-uncased_volcab_size_6000\\added_tokens.json',
 './bert-base-uncased_volcab_size_6000\\tokenizer.json')

3. Preprocess Data

In [9]:
# Load libraries
from transformers import AutoTokenizer
import multiprocessing

In [10]:
# Load the newly trained tokenizer from disk
tokenizer = AutoTokenizer.from_pretrained(f"./{tokenizer_id}" + "_volcab_size_" + str(volcab_size))
# Define number of processes for multiprocessing
num_proc = multiprocessing.cpu_count()
print(f"using {num_proc} processes for tokenization")
print(f"The max length of the tokenizer is {tokenizer.model_max_length}")

using 20 processes for tokenization
The max length of the tokenizer is 512


In [11]:
# Define a tokenization function
def tokenize_text(examples, tokenizer=None):
    return tokenizer(
        examples["text"], # Extract 'text' column from dataset
        return_special_tokens_mask=True, # Required for special pretraining task, returns an additional arr for special tokens such as MASK or SEP.
        truncation=True, # Tell the tokenizer to cut off texts longer than max_length
        max_length=tokenizer.model_max_length, # specifies the max length
    )

In [11]:
# # Use one sample to test the tokenization function
# sample_size = 2
# examples = small_test_dataset.select(range(sample_size))
# # Test tokenization function
# test_tokenized_text_example = examples.map(
#     tokenize_text,
#     batched=True,
#     remove_columns=['text'],
#     num_proc=num_proc,
#     fn_kwargs={'tokenizer': tokenizer},
#     desc="Tokenizing the dataset",
# )
# test_tokenized_text_example.features

In [13]:
# Tokenize dataset with multiprocessing

######### Change dataset here #########
tokenized_datasets = small_test_dataset.map(
    tokenize_text,
    batched=True,
    remove_columns=['text'],
    num_proc=num_proc,
    fn_kwargs={'tokenizer': tokenizer},
    desc="Tokenizing the dataset",
)
tokenized_datasets.features

Tokenizing the dataset (num_proc=20):   0%|          | 0/50000 [00:00<?, ? examples/s]

{'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8')),
 'special_tokens_mask': List(Value('int8'))}

In [14]:
# Shuffle text
tokenized_datasets = tokenized_datasets.shuffle(seed=34)
print(f"the dataset contains in total {len(tokenized_datasets)*tokenizer.model_max_length} tokens")

the dataset contains in total 25600000 tokens


4. Train the model

In [15]:
# import libraries for training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, BertForMaskedLM, BertConfig

In [16]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./bert_pretrained_model',
)

# Load configuration for BERT model
config = BertConfig.from_pretrained('bert-base-uncased')

# Modify the config according to need
# If using the smaller vocab size, the embedding size must be changed accordingly
# e.g. volcab_size = 6000 for our test case
config.vocab_size = volcab_size

# Load model architecture
# Set config=config if using modified config
# BertForMaskedLM is used for masked language modeling task
# It loads the model with random weights if no pretrained weights are specified
# To load pretrained weights, use model = BertForMaskedLM(config=config)
model = BertForMaskedLM(config=config)

# Define data collator for MLM task
# Data collator will dynamically mask tokens during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

In [17]:
# Define Trainer
# A trainer is responsible for the training loop
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator
)

In [19]:
# Start training
trainer.train()

# Save the trained model to disk
trainer.save_model("./bert_pretrained_model_final")

# Save the tokenizer too
tokenizer.save_pretrained("./bert_pretrained_model_final")

Step,Training Loss


KeyboardInterrupt: 